# Explanation of how the notebooks work

There are 4 different notebooks:
1. TAME_DataSelection_MKK
- Here, 20 participants are selected from the TAME dataset based on gender and ethnicity/race.
- From these participants, 1055 files are selected with as balanced a distribution as possible across the pain groups.
2. TAME_Perturbations_MKK
- In this notebook, perturbations are added to the data.
3. TAME_openSMILE_perturbations_MKK
- Here, features are extracted using openSMILE.
- In the bottom code block, this is done separately for the different pain groups (RQ3).
4. TAME_Robustness_evaluation_MKK
- This notebook is used to generate heatmaps to provide an overview of the results.

# TAME Data Selection
*Author: Martine Klep*  
*Date: 03-05-2026*

This script implements a complete pipeline for selecting and preparing audio data from the TAME dataset.

Steps:
1. Load and define files
2. Selection of 20 participants based on gender and race/ethnicity
3. Selection of valid audio files based on: ACTION LABEL = 0 and audiofiles with any content in notes are excluded
4. Balancing the painscores
5. Creation of datasets
   

In [ ]:
#TAME/
#|-- selected_participants_20.csv
#|-- selected_audio_files_20_selection.csv
#|-- selected_audio_files_20_original.csv
#|-- selected_audio_files_20_wiener.csv
#|-- selected_audio_files_20_pain_balanced_selection.csv
#|-- selected_audio_files_20_pain_balanced_original.csv
#|-- selected_audio_files_20_pain_balanced_wiener.csv

# Separate pain-group datasets
#|-- selected_audio_files_20_pain_1_to_4_selection.csv
#|-- selected_audio_files_20_pain_1_to_4_original.csv
#|-- selected_audio_files_20_pain_1_to_4_wiener.csv
#|-- selected_audio_files_20_pain_5_to_6_selection.csv
#|-- selected_audio_files_20_pain_5_to_6_original.csv
#|-- selected_audio_files_20_pain_5_to_6_wiener.csv
#|-- selected_audio_files_20_pain_7_to_10_selection.csv
#|-- selected_audio_files_20_pain_7_to_10_original.csv
#|-- selected_audio_files_20_pain_7_to_10_wiener.csv

# maps for .wav files
#|-- selected_original_audio_20/
#|-- selected_wiener_audio_20/
#|-- selected_original_audio_20_pain_balanced/
#|-- selected_wiener_audio_20_pain_balanced/

# Separate pain-group audio folders
#|-- selected_original_audio_20_pain_1_to_4/
#|-- selected_wiener_audio_20_pain_1_to_4/
#|-- selected_original_audio_20_pain_5_to_6/
#|-- selected_wiener_audio_20_pain_5_to_6/
#|-- selected_original_audio_20_pain_7_to_10/
#|-- selected_wiener_audio_20_pain_7_to_10/

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

# Paths
DATA_PATH = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME")
AUDIO_PATH = DATA_PATH / "mic1_trim_v1~"
AUDIO_BASE_PATH = AUDIO_PATH / "mic1_trim_v2"
META_AUDIO_PATH = DATA_PATH / "meta_audio.csv"
META_PARTICIPANT_PATH = DATA_PATH / "meta_participant.csv"

assert DATA_PATH.exists(), "Data map niet gevonden!"
assert AUDIO_PATH.exists(), "Audio map niet gevonden!"
assert AUDIO_BASE_PATH.exists(), "Audio base path niet gevonden!"
assert META_AUDIO_PATH.exists(), "meta_audio.csv niet gevonden!"
assert META_PARTICIPANT_PATH.exists(), "meta_participant.csv niet gevonden!"

print("The paths are correct")


# Helpers
def load_csv_flexible(file_path):
    """
    Load a CSV file. If it is incorrectly read as a single column,
    split that column into multiple columns based on commas.
    """
    df = pd.read_csv(file_path)

    if df.shape[1] == 1:
        first_col = df.columns[0]
        split_df = df[first_col].str.split(",", expand=True)
        split_df.columns = [col.strip() for col in first_col.split(",")]
        df = split_df.copy()

    df.columns = [col.strip() for col in df.columns]
    return df


def stratified_participant_sample(
    file_path,
    n=20,
    gender_col="GENDER",
    race_col="RACE/ETHNICITY",
    id_col="PID",
    random_state=42
):
    """
    Sample n participants from the meta_participant.csv file, ensuring a balanced distribution.
    The sampling is stratified based on the combination of gender and race/ethnicity. 
    The function first calculates the proportion of each stratum in the available data, then allocates samples accordingly, and finally adjusts for any rounding issues to ensure the total number of samples matches n.
    """

    df = load_csv_flexible(file_path).copy()

    needed_cols = [id_col, gender_col, race_col]
    for col in needed_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available columns: {list(df.columns)}")

    df = df.dropna(subset=[gender_col, race_col]).copy()

    if n > len(df):
        raise ValueError(f"You asked for {n} participants, but only {len(df)} are available.")

    df["stratum"] = (
        df[gender_col].astype(str).str.strip() + " | " +
        df[race_col].astype(str).str.strip()
    )

    stratum_counts = df["stratum"].value_counts().sort_index()
    proportions = stratum_counts / len(df)

    target_counts = proportions * n
    allocated = np.floor(target_counts).astype(int)

    remainder = n - allocated.sum()
    fractional_parts = (target_counts - allocated).sort_values(ascending=False)

    for stratum in fractional_parts.index[:remainder]:
        allocated[stratum] += 1

    allocated = pd.Series({
        stratum: min(allocated[stratum], stratum_counts[stratum])
        for stratum in allocated.index
    })

    current_total = allocated.sum()
    if current_total < n:
        shortfall = n - current_total
        spare_capacity = pd.Series({
            stratum: stratum_counts[stratum] - allocated[stratum]
            for stratum in allocated.index
        }).sort_values(ascending=False)

        for stratum in spare_capacity.index:
            if shortfall == 0:
                break
            if spare_capacity[stratum] > 0:
                extra = min(spare_capacity[stratum], shortfall)
                allocated[stratum] += extra
                shortfall -= extra

    sampled_parts = []
    for stratum, k in allocated.items():
        if k > 0:
            subset = df[df["stratum"] == stratum]
            sampled_parts.append(subset.sample(n=k, random_state=random_state))

    sample_df = (
        pd.concat(sampled_parts)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
        .drop(columns=["stratum"])
    )

    return sample_df


def print_sample_summary(sample_df, gender_col="GENDER", race_col="RACE/ETHNICITY", id_col="PID"):
    """
    Print a summary of the sampled participants, including the total number, distribution of gender and race/ethnicity, and the list of participant IDs.
    """
    print(f"\nNumber of selected participants: {len(sample_df)}\n")
    print("Distribution gender:")
    print(sample_df[gender_col].value_counts(dropna=False))
    print("\nDistribution race/ethnicity:")
    print(sample_df[race_col].value_counts(dropna=False))
    print("\nSelected participants:")
    print(sample_df[id_col].tolist())
    print()


def select_valid_audio_rows(meta_audio_path, selected_participants_df):
    """
    Filter the meta_audio.csv to include only rows for the selected participants, and then further filter to include only valid audio files (NOTES is empty and ACTION LABEL is 0).
    """

    meta_audio_df = load_csv_flexible(meta_audio_path).copy()
    selected_pids = selected_participants_df["PID"].astype(str).tolist()

    filtered_df = meta_audio_df[meta_audio_df["PID"].astype(str).isin(selected_pids)].copy()
    filtered_df["NOTES"] = filtered_df["NOTES"].fillna("").astype(str).str.strip()
    filtered_df["ACTION LABEL"] = pd.to_numeric(filtered_df["ACTION LABEL"], errors="coerce")

    filtered_df = filtered_df[
        (filtered_df["NOTES"] == "") &
        (filtered_df["ACTION LABEL"] == 0)
    ].copy()

    return filtered_df


def balance_audio_by_pain_score_soft(
    df,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
):
    """
    Create a more balanced subset of audio files across pain scores.
    The function limits the number of samples per pain level to reduce class imbalance. 
    If the resulting dataset contains fewer samples than the target total, additional samples are randomly added from the remaining data.
    """
    data = df.copy()
    data[pain_col] = pd.to_numeric(data[pain_col], errors="coerce")
    data = data.dropna(subset=[pain_col]).copy()
    data[pain_col] = data[pain_col].astype(int)

    counts = data[pain_col].value_counts().sort_index()
    print("\nOriginal pain distribution:")
    print(counts)

    n_classes = len(counts)
    ideal_per_class = target_total // n_classes
    max_per_class = int(ideal_per_class * max_ratio)

    sampled_parts = []
    for pain_level, group in data.groupby(pain_col):
        n_samples = min(len(group), max_per_class)
        sampled_parts.append(group.sample(n=n_samples, random_state=random_state))

    balanced_df = pd.concat(sampled_parts)

    if len(balanced_df) < target_total:
        remaining = data.drop(balanced_df.index)
        extra_needed = target_total - len(balanced_df)

        if len(remaining) > 0:
            extra_samples = remaining.sample(
                n=min(extra_needed, len(remaining)),
                random_state=random_state
            )
            balanced_df = pd.concat([balanced_df, extra_samples])

    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nFinal number of samples:", len(balanced_df))
    print("\nNew pain distribution:")
    print(balanced_df[pain_col].value_counts().sort_index())

    return balanced_df


def build_audio_paths(filtered_audio_df, audio_base_path, extension=".wav"):
    """
    Build the expected file paths for the audio files based on the filtered audio DataFrame and the base path.
    The function constructs the filename using the PID, COND, UTTNUM, and UTTID columns, and then creates the full file path. It also checks whether each file exists and adds a boolean column indicating the existence of the file.
    """
    df = filtered_audio_df.copy()
    df["PID"] = df["PID"].astype(str).str.strip()
    df["COND"] = df["COND"].astype(str).str.strip()
    df["UTTNUM"] = df["UTTNUM"].astype(str).str.strip()
    df["UTTID"] = df["UTTID"].astype(str).str.strip()

    df["filename"] = (
        df["PID"] + "." +
        df["COND"] + "." +
        df["UTTNUM"] + "." +
        df["UTTID"] + extension
    )

    df["file_path"] = df.apply(
        lambda row: audio_base_path / row["PID"] / row["filename"],
        axis=1
    )
    df["file_exists"] = df["file_path"].apply(lambda p: p.exists())

    return df


def make_output_path(input_path, output_root):
    """
    Create an output path for the audio file that preserves the participant's directory structure.
    """

    input_path = Path(input_path)
    output_root = Path(output_root)

    participant_id = input_path.parent.name
    filename = input_path.name

    participant_output_dir = output_root / participant_id
    participant_output_dir.mkdir(parents=True, exist_ok=True)

    return participant_output_dir / filename


def copy_selected_audio(audio_selection_df, output_dir):
    """
    Copy the selected audio files to the specified output directory, preserving the participant's directory structure. The function returns the number of files successfully copied.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    copied_count = 0

    for _, row in audio_selection_df.iterrows():
        input_path = Path(row["file_path"])

        try:
            output_path = make_output_path(input_path, output_dir)
            shutil.copy2(input_path, output_path)
            copied_count += 1

        except Exception as e:
            print(f"Error copying {input_path}: {e}")

    return copied_count


def run_dataset_pipeline(
    dataset_name,
    audio_rows_df,
    audio_base_path,
    data_path,
    apply_pain_balance=False,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
):
    """
    Run the full pipeline for a given dataset configuration. This includes optional pain score balancing, building audio paths, checking file existence, and copying existing files to the output directory. The function also prints summary statistics at each step and returns a dictionary with the results.
    """
    if apply_pain_balance:
        audio_rows_df = balance_audio_by_pain_score_soft(
            df=audio_rows_df,
            pain_col=pain_col,
            target_total=target_total,
            max_ratio=max_ratio,
            random_state=random_state
        )

    audio_selection = build_audio_paths(audio_rows_df, audio_base_path=audio_base_path)
    audio_selection_existing = audio_selection[audio_selection["file_exists"]].copy()

    print(f"\n[{dataset_name}] Number of selected audio rows: {len(audio_selection)}")
    print(f"[{dataset_name}] Number of existing files: {audio_selection_existing['file_exists'].sum()}")

    output_dir = data_path / f"original_audiofiles_{dataset_name}"
    copied_count = copy_selected_audio(
        audio_selection_df=audio_selection_existing,
        output_dir=output_dir
    )

    print(f"[{dataset_name}] Saved original folder: {output_dir}")
    print(f"[{dataset_name}] Number of copied files: {copied_count}")

    return {
        "output_dir": output_dir,
        "copied_count": copied_count,
        "n_selected": len(audio_selection),
        "n_existing": int(audio_selection_existing["file_exists"].sum()),
    }


# Run the pipeline for 20 participants
sample_20 = stratified_participant_sample(
    file_path=META_PARTICIPANT_PATH,
    n=20,
    random_state=42
)

print_sample_summary(sample_20)

# Valid audio rows for those 20 participants
filtered_audio_20 = select_valid_audio_rows(
    meta_audio_path=META_AUDIO_PATH,
    selected_participants_df=sample_20
)

# Dataset 20 participants, no pain balancing
results_20 = run_dataset_pipeline(
    dataset_name="20",
    audio_rows_df=filtered_audio_20,
    audio_base_path=AUDIO_BASE_PATH,
    data_path=DATA_PATH,
    apply_pain_balance=False,
    random_state=42
)

# Dataset 20 participants + pain balanced
results_20_pain = run_dataset_pipeline(
    dataset_name="20_pain",
    audio_rows_df=filtered_audio_20,
    audio_base_path=AUDIO_BASE_PATH,
    data_path=DATA_PATH,
    apply_pain_balance=True,
    pain_col="REVISED PAIN",
    target_total=1000,
    max_ratio=1.5,
    random_state=42
)

print("\nFinished.")

The paths are correct

Number of selected participants: 20

Distribution gender:
GENDER
Woman         10
Man            8
Male           1
Non-binary     1
Name: count, dtype: int64

Distribution race/ethnicity:
RACE/ETHNICITY
Asian                11
White                 6
Hispanic/Latino       2
Two or more races     1
Name: count, dtype: int64

Selected participants:
['p80330', 'p28030', 'p60145', 'p64560', 'p79665', 'p68870', 'p68340', 'p77560', 'p59520', 'p79550', 'p91315', 'p15965', 'p68625', 'p18785', 'p97630', 'p10085', 'p20960', 'p62650', 'p37540', 'p94215']


[20] Number of selected audio rows: 1941
[20] Number of existing files: 1941
[20] Saved original folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\original_audiofiles_20
[20] Number of copied files: 1941

Original pain distribution:
REVISED PAIN
1     913
2     107
3     148
4     273
5     122
6     144
7     138
8      71
9      20
10      5
Name: count, dtype: int64



# Pain group splitting for RQ3
This section describes the procedure used to create separate datasets for different pain groups.
The dataset is divided into three pain groups:
- Mild pain: 1-4
- Moderate pain: 5-6
- Severe pain: 7-10

Steps:
1. Split the audio in seperate datasets
2. For each group there is a target of 200 audiofiles

In [45]:
# separate pain groups (1-4, 5-6, 7-10)
def split_audio_by_pain_bins(
    df,
    pain_col="REVISED PAIN",
    bin_targets=None,
    random_state=42
):
    """
    Split audio into separate balanced datasets per pain bin.
    Each bin is sampled as evenly as possible across the pain scores within that bin.
    """
    if bin_targets is None:
        bin_targets = {
            "1-4": {"scores": [1, 2, 3, 4], "target": 200},
            "5-6": {"scores": [5, 6], "target": 200},
            "7-10": {"scores": [7, 8, 9, 10], "target": 200},
        }

    data = df.copy()
    data[pain_col] = pd.to_numeric(data[pain_col], errors="coerce")
    data = data.dropna(subset=[pain_col]).copy()
    data[pain_col] = data[pain_col].astype(int)

    print("\nOriginal pain distribution:")
    print(data[pain_col].value_counts().sort_index())

    split_results = {}

    for bin_name, settings in bin_targets.items():
        scores = settings["scores"]
        target = settings["target"]

        bin_df = data[data[pain_col].isin(scores)].copy()

        if bin_df.empty:
            print(f"\nWarning: no samples available for bin {bin_name}")
            continue

        print(f"\nProcessing bin {bin_name} (target={target})")

        score_groups = {
            score: bin_df[bin_df[pain_col] == score].copy()
            for score in scores
        }

        available_counts = {score: len(group) for score, group in score_groups.items()}
        print("Available counts:", available_counts)

        n_scores = len(scores)
        base_target = target // n_scores
        remainder = target % n_scores

        allocated = {
            score: min(base_target, available_counts[score])
            for score in scores
        }

        spare_capacity = {
            score: available_counts[score] - allocated[score]
            for score in scores
        }

        while remainder > 0:
            candidates = [s for s in scores if spare_capacity[s] > 0]
            if not candidates:
                break

            best_score = max(candidates, key=lambda s: spare_capacity[s])
            allocated[best_score] += 1
            spare_capacity[best_score] -= 1
            remainder -= 1

        current_total = sum(allocated.values())
        shortfall = target - current_total

        if shortfall > 0:
            spare_capacity = {
                score: available_counts[score] - allocated[score]
                for score in scores
            }

            for score in sorted(scores, key=lambda s: spare_capacity[s], reverse=True):
                if shortfall == 0:
                    break
                if spare_capacity[score] > 0:
                    extra = min(spare_capacity[score], shortfall)
                    allocated[score] += extra
                    shortfall -= extra

        print("Allocated counts:", allocated)

        sampled_parts = []
        for score in scores:
            n_take = allocated[score]
            if n_take > 0:
                sampled_parts.append(
                    score_groups[score].sample(n=n_take, random_state=random_state)
                )

        group_df = (
            pd.concat(sampled_parts)
            .sample(frac=1, random_state=random_state)
            .reset_index(drop=True)
        )

        print(f"Final number of samples in {bin_name}: {len(group_df)}")
        print(group_df[pain_col].value_counts().sort_index())

        split_results[bin_name] = group_df

    return split_results


# Run separate pain-group datasets
pain_group_datasets = split_audio_by_pain_bins(
    df=filtered_audio_20,
    pain_col="REVISED PAIN",
    bin_targets={
        "1-4": {"scores": [1, 2, 3, 4], "target": 200},
        "5-6": {"scores": [5, 6], "target": 200},
        "7-10": {"scores": [7, 8, 9, 10], "target": 200},
    },
    random_state=42
)

results_pain_groups = {}

for group_name, group_df in pain_group_datasets.items():
    safe_group_name = group_name.replace("-", "_to_")

    print(f"\n===== Running pain group {group_name} =====")

    results_pain_groups[group_name] = run_dataset_pipeline(
        dataset_name=f"20_pain_{safe_group_name}",
        audio_rows_df=group_df,
        audio_base_path=AUDIO_BASE_PATH,
        data_path=DATA_PATH,
        apply_pain_balance=False,  
        random_state=42
    )

print("\nFinished separate pain-group datasets.")


Original pain distribution:
REVISED PAIN
1     913
2     107
3     148
4     273
5     122
6     144
7     138
8      71
9      20
10      5
Name: count, dtype: int64

Processing bin 1-4 (target=200)
Available counts: {1: 913, 2: 107, 3: 148, 4: 273}
Allocated counts: {1: 50, 2: 50, 3: 50, 4: 50}
Final number of samples in 1-4: 200
REVISED PAIN
1    50
2    50
3    50
4    50
Name: count, dtype: int64

Processing bin 5-6 (target=200)
Available counts: {5: 122, 6: 144}
Allocated counts: {5: 100, 6: 100}
Final number of samples in 5-6: 200
REVISED PAIN
5    100
6    100
Name: count, dtype: int64

Processing bin 7-10 (target=200)
Available counts: {7: 138, 8: 71, 9: 20, 10: 5}
Allocated counts: {7: 125, 8: 50, 9: 20, 10: 5}
Final number of samples in 7-10: 200
REVISED PAIN
7     125
8      50
9      20
10      5
Name: count, dtype: int64

===== Running pain group 1-4 =====

[20_pain_1_to_4] Number of selected audio rows: 200
[20_pain_1_to_4] Number of existing files: 200
[20_pain_1_to_4]